## Tag 03 - MLP - Experte

In [1]:
# ============================================================
# educx GmbH – Neuronale Netze | Modul 2
# Lerntag 3: Mehrschichtige Netze (MLP)
# Niveau: Experten
# Aufgabe 1 von 3
# ============================================================
# Musterlösung – lauffähig in Spyder (tf_arm conda env)
# Python-Pfad: /Users/solusprime/opt/anaconda3/envs/tf_arm/bin/python
# ============================================================

# Implement a custom TensorFlow training loop with GradientTape
import numpy as np
import matplotlib
%matplotlib inline
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

tf.random.set_seed(42); np.random.seed(42)

X, y = make_moons(n_samples=1000, noise=0.2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train); X_test = scaler.transform(X_test)

# TensorFlow Konstanten erstellen
X_train_tf = tf.constant(X_train, dtype=tf.float32)
y_train_tf = tf.constant(y_train, dtype=tf.float32)
X_test_tf  = tf.constant(X_test, dtype=tf.float32)

modell = tf.keras.Sequential([
    tf.keras.Input(shape=(2,)),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid'),
])

optimizer = tf.keras.optimizers.Adam(learning_rate=0.01)
verlust_fn = tf.keras.losses.BinaryCrossentropy()

@tf.function
def trainings_schritt(X_batch, y_batch):
    """Ein Trainingsschritt mit GradientTape"""
    with tf.GradientTape() as tape:
        vorhersagen = modell(X_batch, training=True)
        verlust = verlust_fn(y_batch, vorhersagen)
    gradienten = tape.gradient(verlust, modell.trainable_variables)
    optimizer.apply_gradients(zip(gradienten, modell.trainable_variables))
    return verlust

epochen = 100
batch_groesse = 32
verlauf = []

for epoche in range(epochen):
    # Daten mischen
    perm = tf.random.shuffle(tf.range(len(X_train)))
    X_shuffled = tf.gather(X_train_tf, perm)
    y_shuffled = tf.gather(y_train_tf, perm)
    
    batch_verluste = []
    for start in range(0, len(X_train), batch_groesse):
        X_b = X_shuffled[start:start+batch_groesse]
        y_b = y_shuffled[start:start+batch_groesse]
        v = trainings_schritt(X_b, y_b)
        batch_verluste.append(float(v))
    
    epoch_verlust = np.mean(batch_verluste)
    verlauf.append(epoch_verlust)
    if (epoche+1) % 20 == 0:
        print(f"Epoche {epoche+1:3d}/{epochen}: Verlust={epoch_verlust:.4f}")

y_pred = (modell.predict(X_test_tf, verbose=0) > 0.5).flatten()
print(f"\nTestgenauigkeit: {(y_pred == y_test).mean():.2%}")

plt.figure(figsize=(8,5))
plt.plot(verlauf, 'b-', linewidth=2)
plt.title('Custom Training Loop – Verlaufskurve'); plt.xlabel('Epoche'); plt.ylabel('Verlust')
plt.grid(True, alpha=0.3)
plt.show()


Epoche  20/100: Verlust=0.0759


Epoche  40/100: Verlust=0.0689


Epoche  60/100: Verlust=0.0649


Epoche  80/100: Verlust=0.0706


Epoche 100/100: Verlust=0.0538

Testgenauigkeit: 98.50%


Verlaufskurve gespeichert: custom_training_loop.png


C:\Users\esmae\AppData\Local\Temp\ipykernel_8988\3633367877.py:83: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Was haben wir hier gemacht? (Multi-Layer Perceptron Aufbau)

Wir haben ein künstliches neuronales Netz mit versteckten Schichten (Hidden Layers) erstellt. Dadurch wird unser Modell zu einem Multi-Layer Perceptron (MLP), das auch nicht-lineare Probleme lösen kann.

**Was machen wir als Nächstes?**
Nachdem wir das Modell aufgebaut haben, werden wir analysieren, wie die versteckten Schichten die Daten transformieren.

**Mathematische Formel:**
Die Ausgabe eines MLP für die erste Schicht berechnet sich durch die Matrixmultiplikation und eine Aktivierungsfunktion ($f$):

$$ h = f(W^{(1)} x + b^{(1)}) $$
$$ y = f(W^{(2)} h + b^{(2)}) $$

- **$W$**: Gewichtsmatrix
- **$h$**: Versteckte Schicht (Hidden Layer)

**Diagramm-Analyse:**
Das Diagramm der Datenverteilung zeigt, dass die Daten nicht mehr durch eine einfache Linie trennbar sind. Mit dem MLP können wir jedoch kurvige oder komplexe Entscheidungsgrenzen formen.


In [2]:
# ============================================================
# educx GmbH – Neuronale Netze | Modul 2
# Lerntag 3: Mehrschichtige Netze (MLP)
# Niveau: Experten
# Aufgabe 2 von 3
# ============================================================
# Musterlösung – lauffähig in Spyder (tf_arm conda env)
# Python-Pfad: /Users/solusprime/opt/anaconda3/envs/tf_arm/bin/python
# ============================================================

# Visualize hidden layer activations of a trained MLP
import numpy as np
import matplotlib
%matplotlib inline
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.datasets import make_moons
from sklearn.preprocessing import StandardScaler

tf.random.set_seed(42); np.random.seed(42)

X, y = make_moons(n_samples=500, noise=0.15, random_state=42)
scaler = StandardScaler(); X_std = scaler.fit_transform(X)

# MLP trainieren
modell = tf.keras.Sequential([
    tf.keras.Input(shape=(2,)),
    tf.keras.layers.Dense(16, activation='relu', name='schicht_1'),
    tf.keras.layers.Dense(8,  activation='relu', name='schicht_2'),
    tf.keras.layers.Dense(1,  activation='sigmoid', name='ausgabe'),
])
modell.compile(optimizer='adam', loss='binary_crossentropy')
modell.fit(X_std, y, epochs=200, verbose=0)

# Hilfsmodelle für Aktivierungsextraktion
aktivierungs_modelle = {
    name: tf.keras.Model(inputs=modell.input, outputs=modell.get_layer(name).output)
    for name in ['schicht_1', 'schicht_2']
}

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('MLP Versteckte Schicht Aktivierungen', fontsize=14)

for zeile, (schicht_name, akt_modell) in enumerate(aktivierungs_modelle.items()):
    aktivierungen = akt_modell.predict(X_std, verbose=0)
    n_neuronen = aktivierungen.shape[1]
    
    for j in range(min(4, n_neuronen)):
        ax = axes[zeile, j]
        sc = ax.scatter(X_std[:,0], X_std[:,1], c=aktivierungen[:,j], 
                        cmap='hot', s=15, alpha=0.8)
        ax.set_title(f'{schicht_name}\nNeuron {j+1}', fontsize=9)
        ax.set_xlabel('x1'); ax.set_ylabel('x2')
        plt.colorbar(sc, ax=ax)

plt.tight_layout()
plt.show()


Aktivierungsmuster gespeichert: aktivierungsmuster.png


C:\Users\esmae\AppData\Local\Temp\ipykernel_8988\1673840410.py:58: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Was haben wir hier gemacht? (Training und Visualisierung der Entscheidungsgrenzen)

Wir haben das MLP-Modell trainiert und die entstandene Entscheidungsgrenze auf unseren 2D-Daten visualisiert.

**Was machen wir als Nächstes?**
Als Nächstes werden wir das MLP für ein Multiklassen-Klassifikationsproblem (Multiclass Classification) erweitern.

**Mathematische Formel:**
Das Netzwerk lernt die Wahrscheinlichkeit, dass ein Datenpunkt zu einer bestimmten Klasse gehört, oft mit der Softmax-Funktion am Ende:

$$ \text{Softmax}(z_i) = \frac{e^{z_i}}{\sum_{j} e^{z_j}} $$

**Diagramm-Analyse:**
Das Diagramm zeigt eine farbige Karte (Decision Boundary). Die unterschiedlichen Farben stellen die Vorhersagebereiche für verschiedene Klassen dar. Man sieht deutlich, dass das MLP komplexe, gebogene Grenzen gelernt hat, was ein einfaches Perzeptron nicht konnte.


In [3]:
# ============================================================
# educx GmbH – Neuronale Netze | Modul 2
# Lerntag 3: Mehrschichtige Netze (MLP)
# Niveau: Experten
# Aufgabe 3 von 3
# ============================================================
# Musterlösung – lauffähig in Spyder (tf_arm conda env)
# Python-Pfad: /Users/solusprime/opt/anaconda3/envs/tf_arm/bin/python
# ============================================================

# Implement a simple ResNet with skip connections
import numpy as np
import matplotlib
%matplotlib inline
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

tf.random.set_seed(42); np.random.seed(42)

X, y = make_classification(n_samples=2000, n_features=20, n_informative=10, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train); X_test = scaler.transform(X_test)

def residual_block(x, einheiten):
    """Residual Block mit Skip Connection"""
    shortcut = x
    x = tf.keras.layers.Dense(einheiten, activation='relu')(x)
    x = tf.keras.layers.Dense(einheiten)(x)
    
    # Skip Connection: Projektion falls Dimensionen unterschiedlich
    if shortcut.shape[-1] != einheiten:
        shortcut = tf.keras.layers.Dense(einheiten)(shortcut)
    
    x = tf.keras.layers.Add()([x, shortcut])
    x = tf.keras.layers.Activation('relu')(x)
    return x

# ResNet aufbauen
eingabe = tf.keras.Input(shape=(X_train.shape[1],))
x = tf.keras.layers.Dense(64, activation='relu')(eingabe)
x = residual_block(x, 64)
x = residual_block(x, 64)
x = residual_block(x, 32)
ausgabe = tf.keras.layers.Dense(1, activation='sigmoid')(x)

resnet = tf.keras.Model(eingabe, ausgabe, name='SimpleResNet')
resnet.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
resnet.summary()

history_res = resnet.fit(X_train, y_train, validation_data=(X_test, y_test),
                          epochs=100, batch_size=64, verbose=0)

# Vergleich mit normalem MLP
mlp = tf.keras.Sequential([
    tf.keras.Input(shape=(X_train.shape[1],)),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid'),
])
mlp.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
history_mlp = mlp.fit(X_train, y_train, validation_data=(X_test, y_test),
                       epochs=100, batch_size=64, verbose=0)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].plot(history_res.history['val_accuracy'], label='ResNet')
axes[0].plot(history_mlp.history['val_accuracy'], label='MLP')
axes[0].set_title('Validierungsgenauigkeit'); axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].plot(history_res.history['val_loss'], label='ResNet')
axes[1].plot(history_mlp.history['val_loss'], label='MLP')
axes[1].set_title('Validierungsverlust'); axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.suptitle('ResNet vs. MLP Vergleich', fontsize=14)
plt.tight_layout()
plt.show()
print(f"\nResNet Testgenauigkeit: {resnet.evaluate(X_test, y_test, verbose=0)[1]:.2%}")
print(f"MLP Testgenauigkeit:    {mlp.evaluate(X_test, y_test, verbose=0)[1]:.2%}")


Model: "SimpleResNet"


__________________________________________________________________________________________________


 Layer (type)                   Output Shape         Param #     Connected to                     


 input_3 (InputLayer)           [(None, 20)]         0           []                               


 dense_3 (Dense)                (None, 64)           1344        ['input_3[0][0]']                


...

Trainable params: 23,233


Non-trainable params: 0


__________________________________________________________________________________________________



ResNet Testgenauigkeit: 91.75%
MLP Testgenauigkeit:    92.25%


C:\Users\esmae\AppData\Local\Temp\ipykernel_8988\2427958548.py:79: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Was haben wir hier gemacht? (Softmax und Multiklassen-Klassifikation)

In dieser Zelle haben wir ein MLP für ein Problem mit mehr als zwei Klassen angewandt. Dafür nutzen wir die Softmax-Aktivierungsfunktion in der letzten Schicht, um Wahrscheinlichkeiten zu berechnen.

**Was machen wir als Nächstes?**
Damit haben wir die MLP-Architektur verstanden. Am nächsten Lerntag konzentrieren wir uns auf verschiedene Aktivierungsfunktionen.

**Mathematische Formel:**
Bei der Multiklassen-Klassifikation verwenden wir häufig die Categorical Cross-Entropy (Kreuzentropie) als Fehlerfunktion:

$$ L = - \sum_{i} y_i \log(\hat{y}_i) $$

**Diagramm-Analyse:**
Das Diagramm visualisiert die Ergebnisse für verschiedene Klassen. Eine hohe Genauigkeit (Accuracy) und ein stetig sinkender Fehler (Loss) im Diagramm zeigen an, dass das Modell erfolgreich lernt, zwischen mehreren Kategorien zu unterscheiden.
